# 📊 Project 01 — Customer Retention & Revenue Analytics
## Step 2: Build the Master Analytics Table (MAT)

**Goal:** Join all cleaned tables into **one wide table** — one row per customer — and add new calculated features.

### Why do we need a MAT?
> Right now our data is spread across 7 tables. A machine learning model needs **one row per customer** with ALL their information. The MAT does exactly that.

### How it works:
```
dim_users  +  fact_subscriptions  +  activity_logs (summarised)  +  payments (summarised)  →  MAT ✅
```

---

## 📦 Cell 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

print("✅ Libraries loaded!")

✅ Libraries loaded!


## 📂 Cell 2: Set Folder Paths

In [2]:
# Where cleaned files are (from Step 1)
CLEAN_DIR = "./data/cleaned"

# Where the final MAT will be saved
MAT_DIR = "./data/mat"
os.makedirs(MAT_DIR, exist_ok=True)

print(f"📂 Input  : {CLEAN_DIR}")
print(f"📂 Output : {MAT_DIR}")
print("✅ Folders ready!")

📂 Input  : ./data/cleaned
📂 Output : ./data/mat
✅ Folders ready!


## 📥 Cell 3: Load Cleaned Data

In [3]:
# parse_dates tells pandas to automatically read date columns as dates (not text)
users = pd.read_csv(f"{CLEAN_DIR}/clean_dim_users.csv",
                    parse_dates=["signup_date"])

subs  = pd.read_csv(f"{CLEAN_DIR}/clean_fact_subscriptions.csv",
                    parse_dates=["start_date", "cancel_date"])

pay   = pd.read_csv(f"{CLEAN_DIR}/clean_fact_payments.csv",
                    parse_dates=["payment_date"])

act   = pd.read_csv(f"{CLEAN_DIR}/clean_fact_activity_logs.csv",
                    parse_dates=["event_date"])

print(f"✅ Loaded: {len(users):,} users  |  {len(subs):,} subscriptions")
print(f"           {len(pay):,} payments  |  {len(act):,} activity events")

✅ Loaded: 12,000 users  |  12,000 subscriptions
           170,138 payments  |  1,745,584 activity events


## 🔢 Cell 4: Summarise Activity Logs (Aggregate per User)

### Why summarise?
> The activity table has **1.7 million rows** but only **12,000 customers**.
> We can't put 1.7M rows into one MAT row!
> 
> So we **group by user_id** and create summary stats:
> *"This customer logged in 22 times, used 8 features, averaged 16 min per session"*

In [4]:
print("📊 Summarising activity logs per user...")

# Group all activity rows by user_id and calculate summary stats
act_summary = act.groupby("user_id").agg(

    # Count total events (any login, click, etc.)
    total_events       = ("activity_id",          "count"),

    # Count how many events were logins specifically
    total_logins       = ("event_type",           lambda x: (x == "login").sum()),

    # Count how many DIFFERENT features they used
    distinct_features  = ("feature_used",         "nunique"),

    # Sum of all page views
    total_page_views   = ("page_views",           "sum"),

    # Average session length in minutes
    avg_session_dur    = ("session_duration_min", "mean"),

    # Most recent date they were active
    last_activity_date = ("event_date",           "max"),

    # Total errors they encountered
    total_errors       = ("error_occurred",       "sum"),

    # How many events were on mobile
    mobile_events      = ("is_mobile",            "sum"),

).reset_index()   # Turn the group index back into a normal column

print(f"  ✅ Summarised {len(act):,} activity rows → {len(act_summary):,} user rows")
act_summary.head(3)

📊 Summarising activity logs per user...
  ✅ Summarised 1,745,584 activity rows → 12,000 user rows


,user_id,total_events,total_logins,distinct_features,total_page_views,avg_session_dur,last_activity_date,total_errors,mobile_events
0,USR-00001,200,18,8,2165,0.940000,2024-12-27,8,200
1,USR-00002,200,22,8,1985,1.830000,2024-12-29,7,200
2,USR-00003,60,6,8,631,2.116667,2022-10-21,2,60


## 💳 Cell 5: Summarise Payments (Aggregate per User)

In [5]:
print("💳 Summarising payments per user...")

pay_summary = pay.groupby("user_id").agg(

    # Total number of payment transactions
    total_payments     = ("payment_id",    "count"),

    # Total money spent
    total_revenue      = ("amount_usd",    "sum"),

    # Average payment amount
    avg_payment_amt    = ("amount_usd",    "mean"),

    # How many payments failed
    failed_payments    = ("is_failed",     "sum"),

    # How many payments were refunded
    refunded_payments  = ("is_refunded",   "sum"),

    # Date of most recent payment
    last_payment_date  = ("payment_date",  "max"),

).reset_index()

print(f"  ✅ Summarised {len(pay):,} payment rows → {len(pay_summary):,} user rows")
pay_summary.head(3)

💳 Summarising payments per user...
  ✅ Summarised 170,138 payment rows → 12,000 user rows


,user_id,total_payments,total_revenue,avg_payment_amt,failed_payments,refunded_payments,last_payment_date
0,USR-00001,47,3759.53,79.99,2,0,2024-12-02
1,USR-00002,4,399.84,99.96,0,0,2024-03-06
2,USR-00003,1,99.96,99.96,0,0,2022-03-28


## 🔗 Cell 6: Join All Tables Together

We join the tables using `user_id` as the connecting key.

> **`how='left'`** means: keep ALL rows from the left table, and attach matching data from the right table.
> If no match is found, the row stays but gets NaN for the right-table columns.

In [6]:
print("🔗 Joining all tables together...")

# Step 1: Start with subscriptions (one row per user) + user profile
mat = subs.merge(users, on="user_id", how="left", suffixes=("_sub", "_user"))
print(f"  After joining users      : {mat.shape[0]:,} rows × {mat.shape[1]} columns")

# Step 2: Add activity summary
mat = mat.merge(act_summary, on="user_id", how="left")
print(f"  After joining activity   : {mat.shape[0]:,} rows × {mat.shape[1]} columns")

# Step 3: Add payment summary
mat = mat.merge(pay_summary, on="user_id", how="left")
print(f"  After joining payments   : {mat.shape[0]:,} rows × {mat.shape[1]} columns")

print(f"\n  ✅ Final joined shape: {mat.shape}")

🔗 Joining all tables together...
  After joining users      : 12,000 rows × 37 columns
  After joining activity   : 12,000 rows × 45 columns
  After joining payments   : 12,000 rows × 51 columns

  ✅ Final joined shape: (12000, 51)


## ⚙️ Cell 7: Feature Engineering — Create New Useful Columns

We now calculate extra features that will help the model predict churn.

| New Feature | Formula | Why It Matters |
|---|---|---|
| `days_since_last_login` | snapshot - last activity date | Recently inactive = at risk |
| `avg_monthly_spend` | revenue ÷ (tenure / 30) | Higher spend = more committed |
| `feature_adoption_score` | # distinct features used | Low adoption = not seeing value |
| `login_frequency_monthly` | logins ÷ active months | Logging in less = disengaging |
| `failed_payment_rate` | failed ÷ total payments | Failed payments predict churn |
| `events_per_day` | events ÷ tenure days | Overall activity level |

In [7]:
print("⚙️  Engineering new features...")

# Reference date: pretend today is 2025-01-01 for all recency calculations
SNAPSHOT_DATE = pd.Timestamp("2025-01-01")

# Convert date columns just in case they weren't parsed
mat["last_activity_date"] = pd.to_datetime(mat["last_activity_date"], errors="coerce")
mat["last_payment_date"]  = pd.to_datetime(mat["last_payment_date"],  errors="coerce")
mat["signup_date"]        = pd.to_datetime(mat["signup_date"],        errors="coerce")

# ── Recency Features ────────────────────────────────────
# How many days since they last logged in?
mat["days_since_last_login"]   = (SNAPSHOT_DATE - mat["last_activity_date"]).dt.days

# How many days since they last paid?
mat["days_since_last_payment"] = (SNAPSHOT_DATE - mat["last_payment_date"]).dt.days

# ── Spend Metrics ───────────────────────────────────────
# Average monthly spend — clip(lower=1) avoids dividing by zero
mat["avg_monthly_spend"] = mat["total_revenue"] / (mat["tenure_days"] / 30).clip(lower=1)

# ── Engagement Metrics ──────────────────────────────────
# Feature adoption score: how many different features did they use? (max = 10)
mat["feature_adoption_score"]  = mat["distinct_features"].clip(upper=10)

# How often do they log in per month?
mat["login_frequency_monthly"] = mat["total_logins"] / (mat["tenure_days"] / 30).clip(lower=1)

# What % of their payments failed?
mat["failed_payment_rate"]     = mat["failed_payments"] / mat["total_payments"].clip(lower=1)

# How active are they per day?
mat["events_per_day"]          = mat["total_events"] / mat["tenure_days"].clip(lower=1)

# ── Cohort Month ─────────────────────────────────────────
# Group customers by the month they signed up
# Example: '2022-01' = all customers who joined in January 2022
mat["cohort_month"] = mat["signup_date"].dt.to_period("M").astype(str)

print("  ✅ All new features created!")
print(f"  New features: days_since_last_login, avg_monthly_spend, feature_adoption_score,")
print(f"                login_frequency_monthly, failed_payment_rate, events_per_day, cohort_month")

⚙️  Engineering new features...
  ✅ All new features created!
  New features: days_since_last_login, avg_monthly_spend, feature_adoption_score,
                login_frequency_monthly, failed_payment_rate, events_per_day, cohort_month


## 🎯 Cell 8: RFM Scoring

**RFM = Recency + Frequency + Monetary** — a way to score every customer's health.

Each dimension is scored 1 (worst) to 5 (best):
- **Recency 5** = logged in very recently ✅
- **Frequency 5** = logs in very often ✅  
- **Monetary 5** = high spender ✅

Total RFM Score: 13–15 = **Low Risk** | 8–12 = **Medium Risk** | Below 8 = **High Risk**

In [8]:
print("🎯 Calculating RFM Scores...")

def rfm_score(series, ascending=True):
    """
    Splits customers into 5 equal groups (1=worst, 5=best).
    - ascending=True  → higher value gets higher score (e.g. spend, logins)
    - ascending=False → lower value gets higher score (e.g. days_since_login)
    """
    labels = [1, 2, 3, 4, 5] if ascending else [5, 4, 3, 2, 1]
    return pd.qcut(
        series.rank(method="first"),   # rank handles ties
        q=5,
        labels=labels
    ).astype(int)

# Recency: fewer days since login = better → ascending=False
mat["rfm_recency"]   = rfm_score(mat["days_since_last_login"], ascending=False)

# Frequency: more logins = better → ascending=True
mat["rfm_frequency"] = rfm_score(mat["total_logins"], ascending=True)

# Monetary: higher spend = better → ascending=True
mat["rfm_monetary"]  = rfm_score(mat["avg_monthly_spend"], ascending=True)

# Total RFM score = sum of all three (range: 3 to 15)
mat["rfm_score"] = mat["rfm_recency"] + mat["rfm_frequency"] + mat["rfm_monetary"]

print(f"  ✅ RFM scores calculated")
print(f"  RFM Score range: {mat['rfm_score'].min()} to {mat['rfm_score'].max()}")
print(f"  Average RFM Score: {mat['rfm_score'].mean():.1f}")

🎯 Calculating RFM Scores...
  ✅ RFM scores calculated
  RFM Score range: 3 to 15
  Average RFM Score: 9.0


## 🚦 Cell 9: Assign Risk Segments
Based on the RFM score, classify every customer into a risk category.

In [9]:
print("🚦 Assigning Risk Segments...")

# Define the rules:
# rfm_score >= 11 → Low Risk   (good customers)
# rfm_score  8-10 → Medium Risk
# rfm_score  < 8  → High Risk  (at risk of leaving)

conditions = [
    mat["rfm_score"] >= 11,
    (mat["rfm_score"] >= 8) & (mat["rfm_score"] < 11),
]
choices = ["Low Risk", "Medium Risk"]

# np.select: apply conditions in order, default = "High Risk"
mat["risk_segment"] = np.select(conditions, choices, default="High Risk")

# Show breakdown
seg_counts = mat["risk_segment"].value_counts()
print("\n  Risk Segment Breakdown:")
for seg, count in seg_counts.items():
    pct = count / len(mat) * 100
    print(f"    {seg:<15}: {count:,} customers ({pct:.1f}%)")

🚦 Assigning Risk Segments...

  Risk Segment Breakdown:
    Medium Risk    : 4,550 customers (37.9%)
    Low Risk       : 3,834 customers (31.9%)
    High Risk      : 3,616 customers (30.1%)


## ✅ Cell 10: Final Check & Save

In [10]:
print("📊 Final MAT Summary")
print("=" * 45)
print(f"  Rows (customers) : {mat.shape[0]:,}")
print(f"  Columns (features): {mat.shape[1]}")
print(f"  Churn Rate        : {mat['churned'].mean()*100:.1f}%")
print(f"  High-Risk Users   : {(mat['risk_segment']=='High Risk').sum():,}")

# Save the MAT
output_path = f"{MAT_DIR}/master_analytics_table.csv"
mat.to_csv(output_path, index=False)

print(f"\n  💾 Saved to: {output_path}")
print("\n👉 Next Step: Run step03_eda_cohort_analysis.ipynb")

📊 Final MAT Summary
  Rows (customers) : 12,000
  Columns (features): 64
  Churn Rate        : 23.8%
  High-Risk Users   : 3,616

  💾 Saved to: ./data/mat/master_analytics_table.csv

👉 Next Step: Run step03_eda_cohort_analysis.ipynb


## 👀 Cell 11: Preview the MAT

In [11]:
# Show the first 3 rows with a selection of key columns
key_cols = [
    "user_id", "plan_name", "churned", "risk_segment", "rfm_score",
    "tenure_days", "mrr_usd", "total_logins", "feature_adoption_score",
    "days_since_last_login", "failed_payment_rate", "cohort_month"
]

# Show only columns that actually exist in our MAT
available_cols = [c for c in key_cols if c in mat.columns]
mat[available_cols].head(5)

,user_id,churned,risk_segment,rfm_score,tenure_days,mrr_usd,total_logins,feature_adoption_score,days_since_last_login,failed_payment_rate,cohort_month
0,USR-00001,0,Medium Risk,10,1409,79.99,18,8,5,0.042553,2021-02
1,USR-00002,0,Medium Risk,9,1395,8.33,22,8,3,0.000000,2021-03
2,USR-00003,1,High Risk,5,208,8.33,6,8,803,0.000000,2022-03
3,USR-00004,0,Medium Risk,10,1271,9.99,23,8,4,0.023256,2021-07
4,USR-00005,0,Medium Risk,9,1299,24.99,23,8,8,0.250000,2021-06
